# Serial dependence — does last week predict this week?

*Read-only informative artifact, and a **framing + deferred stub**. This
notebook states the week-to-week-dependence questions and frames the measure
that will answer them, but runs **no analysis** — the statistical computation
it needs (lag-1 within-player autocorrelation) is not yet a kernel, and that
computation belongs in a kernel, not inline in an informative notebook. No gate
decisions, no PROCEED/STOP verdict.*

## Questions a manager asks about week-to-week scoring

- **Does last week predict this week?** Does a haul tend to follow a haul, and
  a blank follow a blank — or is each gameweek roughly a fresh draw?
- **Does "form" carry real signal, or is week-to-week scoring close to
  independent?** Captaincy and transfer timing lean heavily on the assumption
  that recent returns persist; this notebook is where that assumption gets
  tested directly.
- **Does any persistence depend on position?** A clean-sheet-bound keeper and a
  haul-prone forward may carry very different week-to-week memory.

`target_stability` and `signal_stability` ask whether the *season-pooled*
picture drifts across coarse blocks. This notebook asks the finer, sequential
question — **memory between adjacent gameweeks within a single player** — which
neither block view can answer.

## (a) Lag-1 within-player autocorrelation of `total_points` — what it WILL measure

**What we measure** *(planned)* — for each player, the rank correlation between
their `total_points` in consecutive gameweeks they featured in (week *t* vs
week *t+1*), aggregated **within position**. Concretely: build per-player
ordered (gw, total_points) sequences over the participation population, form
lag-1 pairs, and summarise the within-player serial correlation per position
(with sign and spread), pooling players within a position. Later, the same
treatment extends to the candidate signals (does a high-`xg` week predict
another high-`xg` week?).

**What it means** — a clear positive lag-1 autocorrelation is the empirical
footprint of **form**: recent returns carry information about the next return,
so captaincy/transfer timing on recent output is justified. A near-zero
autocorrelation says week-to-week scoring is close to independent and "form"
narratives are mostly noise — a manager should weight season-level quality over
last week's haul.

**What it doesn't mean** — lag-1 autocorrelation is *within-player sequential
memory*, **not** the season-block drift that `target_stability` /
`signal_stability` measure, and **not** a predictive model of next week's
points. It will be computed over the **participation** population
(`minutes > 0`); the 60-minute boundary stays deferred to `population/`. DGW
rows place two fixtures at one (player, gw) point, so the lag structure across a
DGW is itself confounded — a treatment for the `fixture/` layer. Positive
autocorrelation also need not be causal "momentum"; persistent player quality
(a good player scores adjacent weeks) produces it without any week-to-week
carryover.

**Guiding question** — *Does last week predict this week, and does "form" carry
real signal or is scoring close to week-to-week independent?*

## DEFERRED — pending kernel: `diagnostic/serial`

This section will compute the **lag-1 within-player autocorrelation** described
above — first of `total_points`, later of the candidate signals — aggregated by
position over the participation population. Planned shape: build per-player
gameweek-ordered sequences, form adjacent (t, t+1) pairs *within* each player,
compute a rank-based lag-1 serial correlation, and summarise the distribution
of that statistic across players within each position (central tendency, sign,
spread), with the same min-observations and NaN-handling guarantees the other
`research.kernels` diagnostics carry (e.g. `compute_pairwise_rho`).

**Why it is deferred.** This is a real statistical computation — per-player
sequence construction, lag pairing, and a rank correlation aggregated across a
panel — and in this codebase such computations live in `research.kernels`, not
inline in an informative notebook. The kernel
(`research.kernels.diagnostic.serial`) does not exist yet. Until it does, **no
analysis code runs here**: writing the autocorrelation inline would duplicate
logic that belongs in a tested, reusable kernel and would break the
notebook-is-a-thin-consumer convention the foundation layer follows.

When the kernel lands, this stub becomes the consumer: import it, run it over
`df` by position, display the per-position serial-correlation summary, and add
a plain-language closing read — exactly as `target_stability` and
`signal_stability` consume the descriptive/diagnostic kernels today.

## Closing note

No findings yet — this notebook is intentionally a **framing + deferred stub**.
It fixes the week-to-week-dependence questions and frames the lag-1
within-player autocorrelation that will answer them, but the measure itself
waits on `research.kernels.diagnostic.serial`. Until that kernel exists this
notebook carries **no analysis and no verdict**; it is the placeholder
(and the consumer-to-be) for the serial-dependence read, kept beside
`target_stability` and `signal_stability` so the temporal layer's three
questions — block drift of Y, block drift of X, and sequential memory — sit
together.